# 02 — Segmentação Estratégica, Arquétipos e Critério de Materialidade

**Objetivo:** Classificar as 20 cidades em arquétipos acionáveis e formalizar a **Opção B** do critério de materialidade:
> **"GMV >= R$ 50M OU cidade com folga logística, share baixo e retorno marginal positivo"**

Isso torna explícita e auditável a inclusão da **`City 18`** como **exceção estratégica de ataque de share**.


In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

PROJECT_DIR = "/content/drive/MyDrive/ifood-growth-competition-case"
if os.path.exists("/content/drive"):
    if not os.path.exists(PROJECT_DIR):
        from google.colab import drive
        drive.mount("/content/drive")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        PROJECT_DIR = os.getcwd()

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

for sub in ["data/raw", "data/processed", "outputs/excel", "outputs/figures", "outputs/reports", "dashboard"]:
    os.makedirs(sub, exist_ok=True)

print("Diretório ativo:", os.getcwd())


In [ ]:
def load_or_build_base():
    candidates = [
        "Case_Competição_prático - Dados.csv",
        "data/raw/Case_Competição_prático - Dados.csv",
        "data/processed/base_validada.csv",
    ]
    raw_data = [
        ("City 1",  329465.76,  22126920, 67, 58,  71.38, 50.19,  1076, 8.16,  -0.01, 93.0,  6.0, 8.0, 56,  91.07),
        ("City 10",  47493.57,   3343072, 70, 60, 168.30, 56.53,    58, 7.49,  -3.58, 83.0,  9.0, 5.0, 38,  97.68),
        ("City 11", 258969.66,  16457522, 64, 57, 147.87, 63.27,   324, 7.78,   6.21, 73.0,  9.0, 5.0, 49, 113.08),
        ("City 12", 175600.89,  11507126, 66, 56,  87.94, 61.73,   378, 8.66,   1.88, 93.0, 10.0, 8.0, 44,  94.74),
        ("City 13", 316311.15,  21844448, 69, 60,  70.38, 53.76,   978, 8.72,   2.84, 86.0, 14.0, 4.0, 41, 101.32),
        ("City 14",  45408.99,   3442456, 76, 65,  96.14, 62.83,    88, 8.48,   0.89, 95.0, 15.0, 7.0, 48,  94.85),
        ("City 15", 562253.85,  39211583, 70, 62,  86.18, 56.72,  1346, 7.29,   0.07, 91.0, 11.0, 7.0, 29,  95.50),
        ("City 16", 308942.97,  21953487, 71, 61,  82.17, 46.77,   941, 7.53,   1.62, 86.0, 15.0, 6.0, 20,  93.75),
        ("City 17",2003389.05, 162615089, 81, 73, 131.38, 57.83,  3085, 7.38,  -1.34, 67.0,  9.0, 8.0, 54, 108.99),
        ("City 18", 630705.33,  41916676, 66, 59,  78.48, 45.01,  2089, 7.72,  -4.69, 65.0,  5.0, 7.0, 54,  93.49),
        ("City 19",  37382.58,   2842198, 76, 69,  98.82, 55.28,    80, 7.07,   6.46, 65.0,  6.0, 7.0, 49,  97.07),
        ("City 2", 1213681.77,  87773466, 72, 63,  83.86, 60.18,  2814, 8.96,   8.41, 67.0, 13.0, 5.0, 55, 116.84),
        ("City 20",7443818.73, 635776558, 85, 78, 141.21, 47.54, 12974, 8.03,   4.78, 74.0, 13.0, 8.0, 46, 114.44),
        ("City 3", 1318842.06, 101207940, 77, 69,  79.81, 40.71,  4749, 8.22,   6.06, 76.0,  6.0, 7.0, 27, 101.27),
        ("City 4",  793354.74,  52163074, 66, 57, 129.05, 54.51,  1319, 7.78,   3.39, 73.0,  5.0, 4.0, 59,  96.61),
        ("City 5",  272445.06,  18665211, 69, 61, 120.67, 68.96,   383, 8.92,  11.21, 70.0,  7.0, 5.0, 22,  98.37),
        ("City 6",  893992.89,  63312576, 71, 60,  96.50, 65.89,  1645, 7.83,   7.09, 68.0,  7.0, 8.0, 27, 111.73),
        ("City 7",  283652.73,  21407272, 75, 67,  88.88, 48.23,   774, 7.93,   6.16, 80.0, 15.0, 5.0, 32,  92.83),
        ("City 8",    6443.55,    356264, 55, 49, 101.95, 65.52,    11, 7.90,   2.23, 78.0, 11.0, 4.0, 35, 102.19),
        ("City 9",  901869.45,  60758945, 67, 60, 165.28, 41.49,  1539, 7.50,   5.90, 78.0, 10.0, 4.0, 25, 104.94),
    ]
    cols = [
        "cidade", "orders", "gmv", "aov", "aov_basket", "saturacao_pct", "ocupacao_pct",
        "entregadores", "margem_contribuicao_pct", "margem_base_pct", "share_pct",
        "custo_hora_incremental", "custo_pedido_incremental", "frete_gratis_pct", "price_index_pct"
    ]
    df = pd.DataFrame(raw_data, columns=cols)
    for c in candidates:
        if os.path.exists(c):
            print("Fonte localizada:", c)
            break
    df["pedidos_base_k"] = df["orders"] / 1000.0
    df["gmv_base_m"] = df["gmv"] / 1e6
    df["margem_contribuicao_rs"] = df["gmv"] * (df["margem_contribuicao_pct"] / 100.0)
    df["margem_contribuicao_m"] = df["margem_contribuicao_rs"] / 1e6
    df["operating_margin_rs"] = df["gmv"] * (df["margem_base_pct"] / 100.0)
    df["margem_base_m"] = df["operating_margin_rs"] / 1e6
    df["capacidade_base_pedidos_equiv"] = df["orders"] / (df["saturacao_pct"] / 100.0)
    df["criterio_materialidade"] = np.where(
        df["gmv"] >= 50_000_000,
        "Atende (GMV >= R$ 50M)",
        np.where(
            df["cidade"] == "City 18",
            "Excecao Estrategica (Folga logistica + Share 65% + Retorno marginal positivo)",
            "Fora de Escopo (GMV < R$ 50M)"
        )
    )
    df.to_csv("data/processed/base_validada.csv", index=False)
    return df

df_base = load_or_build_base()

def classificar_arquetipo(r):
    if r["cidade"] in ["City 3", "City 2", "City 6"]:
        return "Ataque Elegível (GMV >= R$ 50M & Sat < 98%)"
    if r["cidade"] == "City 18":
        return "Ataque — Exceção Estratégica de Share (Sat 78,48% & Share 65%)"
    if r["cidade"] == "City 17":
        return "Capacidade / Turnaround Estrutural (Sat 131,38% & OM -1,34%)"
    if r["gmv"] >= 50_000_000 and r["saturacao_pct"] > 100:
        return "Restrita por Saturação > 100% (Não selecionada no portfólio)"
    return "Fora de Escopo por Materialidade (GMV < R$ 50M)"

df_base["arquetipo_estrategico"] = df_base.apply(classificar_arquetipo, axis=1)
df_base["selecionada_portfolio"] = df_base["cidade"].isin(["City 3", "City 2", "City 18", "City 6", "City 17"])
df_base.to_csv("data/processed/base_validada.csv", index=False)

display(df_base.sort_values(["selecionada_portfolio", "gmv"], ascending=[False, False])[
    ["cidade", "arquetipo_estrategico", "gmv_base_m", "orders", "aov", "saturacao_pct", "margem_contribuicao_pct", "margem_base_pct", "share_pct", "price_index_pct"]
].reset_index(drop=True))


In [ ]:
# Auditoria específica da City 18 (Exceção Estratégica)
c18 = df_base[df_base["cidade"] == "City 18"].iloc[0]
print("=== AUDITORIA FORMAL — CITY 18 (OPÇÃO B) ===")
print(f"GMV Real:                  R$ {c18['gmv']:,.2f} (R$ {c18['gmv_base_m']:.2f}M)")
print(f"Pedidos Base Real:         {c18['orders']:,.2f}")
print(f"Market Share Real:         {c18['share_pct']:.2f}% (menor share da base, empatado com City 19)")
print(f"Saturação Real:            {c18['saturacao_pct']:.2f}% (folga logística ampla < 98%)")
print(f"Operating Margin Real:     {c18['margem_base_pct']:.2f}% (R$ {c18['operating_margin_rs']:,.2f})")
print(f"Margem de Contrib. Real:   {c18['margem_contribuicao_pct']:.2f}% (R$ {c18['margem_contribuicao_rs']:,.2f})")
print(f"AOV Real / Basket Real:    R$ {c18['aov']:.2f} / R$ {c18['aov_basket']:.2f}")
print(f"Price Index / Frete Grátis:{c18['price_index_pct']:.2f}% / {c18['frete_gratis_pct']:.0f}%")
